# Machine Learning-Based Classification of Potentially Hazardous Near-Earth Objects

## Complete reproducible analysis

This notebook runs the M1-M6 missing-diameter experiments, M7 orbital-only baseline, H-conditional and discovery-era missingness analyses, MOID rule audit, JPL checks for the 17 MOID disagreements, the 33 PHA-labelled records with H > 22, the Query API cross-check, H-uncertainty analysis, consolidated audit, and repeated-split/bootstrap evaluation.
The dataset is loaded from the project GitHub repository. The 33-record audit input and the 17-record verification reference are also read from the repository where needed. No manual dataset upload is required.

In [1]:
# ============================================================
# NEO PROJECT: REPRODUCIBLE EVALUATION OF SIX MISSING-DATA METHODS
# ============================================================

# 1. IMPORT LIBRARIES
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)


# 2. CONFIGURATION
RANDOM_STATE = 42
TEST_SIZE = 0.20

FEATURES = ['a', 'e', 'i', 'om', 'w', 'diameter']
ORBITAL_FEATURES = ['a', 'e', 'i', 'om', 'w']

OUTPUT_DIR = '/content/neo_evaluation_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. LOAD DATASET DIRECTLY FROM GITHUB
DATA_URL = (
    "https://raw.githubusercontent.com/meherhasija/"
    "neo-missingness-lab/refs/heads/main/"
    "data/neo_asteroid_dataset.csv"
)

df = pd.read_csv(DATA_URL)

print("=" * 65)
print("DATASET INFORMATION")
print("=" * 65)
print("Dataset shape:", df.shape)
print("Missing diameter:", df['diameter'].isna().sum())
print("Missing PHA labels:", df['pha'].isna().sum())

# 4. DATA CLEANING
df = df.dropna(subset=['pha']).copy()

df['pha'] = df['pha'].astype(str).str.strip().str.upper()
df = df[df['pha'].isin(['Y', 'N'])].copy()

df['target'] = df['pha'].map({'N': 0, 'Y': 1})

X = df[FEATURES].copy()
y = df['target'].copy()

# 5. COMMON STRATIFIED TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTRAIN-TEST SPLIT")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Test PHA records:", int(y_test.sum()))

# Preserve original test indices
test_indices = X_test.index

# 6. HELPER FUNCTION: MODEL
def create_model():
    return HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        random_state=RANDOM_STATE,
        class_weight='balanced',
        early_stopping=False
    )

# 7. HELPER FUNCTION: METRICS
def calculate_metrics(y_true, predictions, probabilities):
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    return {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(
            y_true, predictions, zero_division=0
        ),
        'Recall': recall_score(
            y_true, predictions, zero_division=0
        ),
        'F1': f1_score(
            y_true, predictions, zero_division=0
        ),
        'ROC_AUC': roc_auc_score(y_true, probabilities),
        'Average_Precision': average_precision_score(
            y_true, probabilities
        ),
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    }

# 8. STORAGE FOR RESULTS
all_predictions = []
all_metrics = []

# 9. METHOD 1: NATIVE MISSING VALUE HANDLING
print("\nRunning M1: Native Missing Value Handling")

model = create_model()
model.fit(X_train, y_train)

prob = model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M1_Native',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 10. METHOD 2: MEDIAN IMPUTATION
print("Running M2: Median Imputation")

median_imputer = SimpleImputer(strategy='median')

X_train_median = pd.DataFrame(
    median_imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index
)

X_test_median = pd.DataFrame(
    median_imputer.transform(X_test),
    columns=FEATURES,
    index=X_test.index
)

model = create_model()
model.fit(X_train_median, y_train)

prob = model.predict_proba(X_test_median)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M2_Median',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 11. METHOD 3: MEDIAN IMPUTATION + MISSINGNESS INDICATOR
print("Running M3: Median + Missingness Indicator")

X_train_m3 = X_train.copy()
X_test_m3 = X_test.copy()

X_train_m3['diameter_missing'] = (
    X_train['diameter'].isna().astype(int)
)
X_test_m3['diameter_missing'] = (
    X_test['diameter'].isna().astype(int)
)

m3_features = FEATURES + ['diameter_missing']

m3_imputer = SimpleImputer(strategy='median')

X_train_m3 = pd.DataFrame(
    m3_imputer.fit_transform(X_train_m3),
    columns=m3_features,
    index=X_train.index
)

X_test_m3 = pd.DataFrame(
    m3_imputer.transform(X_test_m3),
    columns=m3_features,
    index=X_test.index
)

model = create_model()
model.fit(X_train_m3, y_train)

prob = model.predict_proba(X_test_m3)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M3_Median_Indicator',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 12. METHOD 4: KNN IMPUTATION
print("Running M4: KNN Imputation")

# Fit orbital scaler on training data only
orbital_scaler = StandardScaler()

train_orbit_scaled = orbital_scaler.fit_transform(
    X_train[ORBITAL_FEATURES]
)

test_orbit_scaled = orbital_scaler.transform(
    X_test[ORBITAL_FEATURES]
)

# Training records with known diameter are donors
donor_mask = X_train['diameter'].notna()

donor_orbit = train_orbit_scaled[donor_mask.values]
donor_diameter = X_train.loc[
    donor_mask, 'diameter'
].values

if len(donor_diameter) < 5:
    raise ValueError("Not enough known training diameters for KNN.")

knn = NearestNeighbors(n_neighbors=5)
knn.fit(donor_orbit)

def knn_fill_diameter(data, scaled_orbit):
    result = data.copy()
    missing_mask = result['diameter'].isna()

    if missing_mask.any():
        distances, indices = knn.kneighbors(
            scaled_orbit[missing_mask.values]
        )

        neighbor_values = donor_diameter[indices]
        result.loc[missing_mask, 'diameter'] = (
            neighbor_values.mean(axis=1)
        )

    return result

X_train_knn = knn_fill_diameter(
    X_train, train_orbit_scaled
)

X_test_knn = knn_fill_diameter(
    X_test, test_orbit_scaled
)

model = create_model()
model.fit(X_train_knn, y_train)

prob = model.predict_proba(X_test_knn)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M4_KNN',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 13. METHOD 5: ITERATIVE IMPUTATION
print("Running M5: Iterative Imputation")

# Scaling and imputation are fitted using training data only
iterative_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('imputer', IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        random_state=RANDOM_STATE,
        skip_complete=True
    ))
])

X_train_iterative = pd.DataFrame(
    iterative_pipeline.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index
)

X_test_iterative = pd.DataFrame(
    iterative_pipeline.transform(X_test),
    columns=FEATURES,
    index=X_test.index
)

model = create_model()
model.fit(X_train_iterative, y_train)

prob = model.predict_proba(X_test_iterative)[:, 1]
pred = (prob >= 0.5).astype(int)

all_predictions.append(pd.DataFrame({
    'method': 'M5_Iterative',
    'row_index': test_indices,
    'actual': y_test.values,
    'predicted': pred,
    'probability': prob
}))

# 14. METHOD 6: COMPLETE-CASE ANALYSIS
print("Running M6: Complete-Case Analysis")

train_complete_mask = X_train.notna().all(axis=1)
test_complete_mask = X_test.notna().all(axis=1)

X_train_complete = X_train.loc[train_complete_mask]
y_train_complete = y_train.loc[train_complete_mask]

X_test_complete = X_test.loc[test_complete_mask]
y_test_complete = y_test.loc[test_complete_mask]

model = create_model()
model.fit(X_train_complete, y_train_complete)

prob = model.predict_proba(X_test_complete)[:, 1]
pred = (prob >= 0.5).astype(int)

m6_predictions = pd.DataFrame({
    'method': 'M6_Complete_Case',
    'row_index': X_test_complete.index,
    'actual': y_test_complete.values,
    'predicted': pred,
    'probability': prob
})

all_predictions.append(m6_predictions)

print("M6 training records:", len(X_train_complete))
print("M6 testing records:", len(X_test_complete))

# 15. COMBINE PREDICTIONS
predictions_df = pd.concat(
    all_predictions, ignore_index=True
)

# 16. DEFINE TEST SUBGROUPS
measured_indices = X_test.index[
    X_test['diameter'].notna()
]

missing_indices = X_test.index[
    X_test['diameter'].isna()
]

subgroups = {
    'Full_Test_Set': list(X_test.index),
    'Measured_Diameter': list(measured_indices),
    'Missing_Diameter': list(missing_indices)
}

# 17. EVALUATE EVERY METHOD ON EACH VALID SUBGROUP
methods = predictions_df['method'].unique()


for method in methods:
    method_data = predictions_df[
        predictions_df['method'] == method
    ]

    for subgroup_name, subgroup_indices in subgroups.items():

        # M6 is a complete-case model.
        # It must not be reported as evaluating the full test set.
        if method == 'M6_Complete_Case':
            if subgroup_name == 'Full_Test_Set':
                continue

            if subgroup_name == 'Missing_Diameter':
                continue

        subset = method_data[
            method_data['row_index'].isin(subgroup_indices)
        ]

        if len(subset) == 0:
            continue

        y_true = subset['actual']
        y_pred = subset['predicted']
        y_prob = subset['probability']

        metrics = calculate_metrics(
            y_true, y_pred, y_prob
        )

        metrics['Method'] = method
        metrics['Evaluation_Subset'] = subgroup_name
        metrics['Records'] = len(subset)
        metrics['PHA_Prevalence'] = y_true.mean()

        all_metrics.append(metrics)

metrics_df = pd.DataFrame(all_metrics)

# Organize columns
column_order = [
    'Method', 'Evaluation_Subset', 'Records',
    'PHA_Prevalence', 'Accuracy', 'Precision',
    'Recall', 'F1', 'ROC_AUC', 'Average_Precision',
    'TN', 'FP', 'FN', 'TP'
]

metrics_df = metrics_df[column_order]

# 18. SAVE OUTPUT FILES
predictions_path = os.path.join(
    OUTPUT_DIR, 'all_methods_predictions.csv'
)

metrics_path = os.path.join(
    OUTPUT_DIR, 'all_methods_metrics.csv'
)

predictions_df.to_csv(predictions_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

# Separate subgroup metrics
for subgroup in metrics_df['Evaluation_Subset'].unique():
    subgroup_df = metrics_df[
        metrics_df['Evaluation_Subset'] == subgroup
    ]
    subgroup_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f'{subgroup.lower()}_metrics.csv'
        ),
        index=False
    )

# 19. SAVE DATASET AND SPLIT INFORMATION
split_info = pd.DataFrame({
    'row_index': test_indices,
    'actual': y_test.values,
    'diameter_available': X_test['diameter'].notna().values
})

split_info.to_csv(
    os.path.join(OUTPUT_DIR, 'test_split_information.csv'),
    index=False
)

# 20. DISPLAY RESULTS
print("\n" + "=" * 80)
print("FINAL RESULTS: ALL METHODS AND TEST SUBGROUPS")
print("=" * 80)

display(
    metrics_df.round(4).sort_values(
        ['Evaluation_Subset', 'Method']
    ).reset_index(drop=True)
)

print("\nTEST SUBSET COUNTS")
print("Full test set:", len(X_test))
print("Measured diameter:", len(measured_indices))
print("Missing diameter:", len(missing_indices))

print("\nMISSINGNESS DISTRIBUTION")
print("Training diameter missing:",
      X_train['diameter'].isna().sum())
print("Testing diameter missing:",
      X_test['diameter'].isna().sum())

print('M1-M6 evaluation completed. Additional analyses continue below.')


DATASET INFORMATION
Dataset shape: (42815, 21)
Missing diameter: 41569
Missing PHA labels: 135

TRAIN-TEST SPLIT
Training records: 34144
Testing records: 8536
Test PHA records: 510

Running M1: Native Missing Value Handling
Running M2: Median Imputation
Running M3: Median + Missingness Indicator
Running M4: KNN Imputation
Running M5: Iterative Imputation
Running M6: Complete-Case Analysis
M6 training records: 999
M6 testing records: 247

FINAL RESULTS: ALL METHODS AND TEST SUBGROUPS


,Method,Evaluation_Subset,Records,PHA_Prevalence,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision,TN,FP,FN,TP
0,M1_Native,Full_Test_Set,8536,0.0597,0.7733,0.1631,0.6765,0.2629,0.8135,0.2653,6256,1770,165,345
1,M2_Median,Full_Test_Set,8536,0.0597,0.7691,0.1604,0.6765,0.2593,0.8155,0.2628,6220,1806,165,345
2,M3_Median_Indicator,Full_Test_Set,8536,0.0597,0.7700,0.1584,0.6608,0.2556,0.8129,0.2781,6236,1790,173,337
3,M4_KNN,Full_Test_Set,8536,0.0597,0.7623,0.1511,0.6451,0.2449,0.7937,0.2028,6178,1848,181,329
4,M5_Iterative,Full_Test_Set,8536,0.0597,0.7649,0.1555,0.6627,0.2520,0.8030,0.2068,6191,1835,172,338
5,M1_Native,Measured_Diameter,247,0.2996,0.6113,0.4295,0.9054,0.5826,0.8002,0.6487,84,89,7,67
6,M2_Median,Measured_Diameter,247,0.2996,0.5870,0.4125,0.8919,0.5641,0.7820,0.6086,79,94,8,66
7,M3_Median_Indicator,Measured_Diameter,247,0.2996,0.6073,0.4286,0.9324,0.5872,0.8103,0.6695,81,92,5,69
8,M4_KNN,Measured_Diameter,247,0.2996,0.6235,0.4128,0.6081,0.4918,0.7081,0.4540,109,64,29,45
9,M5_Iterative,Measured_Diameter,247,0.2996,0.6316,0.4341,0.7568,0.5517,0.7264,0.4685,100,73,18,56



TEST SUBSET COUNTS
Full test set: 8536
Measured diameter: 247
Missing diameter: 8289

MISSINGNESS DISTRIBUTION
Training diameter missing: 33145
Testing diameter missing: 8289
M1-M6 evaluation completed. Additional analyses continue below.


In [2]:
# ============================================================
# M7: ORBITAL-ONLY BASELINE
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=" * 65)
print("M7: ORBITAL-ONLY BASELINE")
print("=" * 65)

# 1. SELECT ONLY ORBITAL FEATURES
ORBITAL_FEATURES = ['a', 'e', 'i', 'om', 'w']

X_train_orbit = X_train[ORBITAL_FEATURES].copy()
X_test_orbit = X_test[ORBITAL_FEATURES].copy()

# 2. CREATE MODEL
orbital_model = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced',
    early_stopping=False
)

# 3. TRAIN MODEL
orbital_model.fit(X_train_orbit, y_train)

print("Model trained successfully!")

# 4. PREDICT ON FULL TEST SET
orbital_prob = orbital_model.predict_proba(
    X_test_orbit
)[:, 1]

orbital_pred = (orbital_prob >= 0.5).astype(int)

# 5. FUNCTION TO CALCULATE METRICS
def evaluate_orbital_subset(indices, subset_name):

    subset_mask = X_test.index.isin(indices)

    actual = y_test.loc[subset_mask]
    predicted = orbital_pred[subset_mask]
    probabilities = orbital_prob[subset_mask]

    tn, fp, fn, tp = confusion_matrix(
        actual, predicted, labels=[0, 1]
    ).ravel()

    return {
        'Method': 'M7_Orbital_Only',
        'Evaluation_Subset': subset_name,
        'Records': len(actual),
        'PHA_Prevalence': actual.mean(),
        'Accuracy': accuracy_score(actual, predicted),
        'Precision': precision_score(
            actual, predicted, zero_division=0
        ),
        'Recall': recall_score(
            actual, predicted, zero_division=0
        ),
        'F1': f1_score(
            actual, predicted, zero_division=0
        ),
        'ROC_AUC': roc_auc_score(
            actual, probabilities
        ),
        'Average_Precision': average_precision_score(
            actual, probabilities
        ),
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    }

# 6. EVALUATE THREE TEST SUBSETS
orbital_results = []

orbital_results.append(
    evaluate_orbital_subset(
        X_test.index,
        'Full_Test_Set'
    )
)

orbital_results.append(
    evaluate_orbital_subset(
        X_test.index[X_test['diameter'].notna()],
        'Measured_Diameter'
    )
)

orbital_results.append(
    evaluate_orbital_subset(
        X_test.index[X_test['diameter'].isna()],
        'Missing_Diameter'
    )
)

orbital_metrics = pd.DataFrame(orbital_results)

# 7. SAVE ORBITAL-ONLY PREDICTIONS
orbital_predictions = pd.DataFrame({
    'method': 'M7_Orbital_Only',
    'row_index': X_test.index,
    'actual': y_test.values,
    'predicted': orbital_pred,
    'probability': orbital_prob
})

# 8. SAVE RESULTS
output_dir = '/content/neo_evaluation_results'
os.makedirs(output_dir, exist_ok=True)

orbital_metrics.to_csv(
    os.path.join(
        output_dir,
        'orbital_only_metrics.csv'
    ),
    index=False
)

orbital_predictions.to_csv(
    os.path.join(
        output_dir,
        'orbital_only_predictions.csv'
    ),
    index=False
)

# 9. DISPLAY RESULTS
print("\nORBITAL-ONLY RESULTS")
display(orbital_metrics.round(4))

print("\nPrediction records:", len(orbital_predictions))
print("Features used:", ORBITAL_FEATURES)

print("\nM7 evaluation completed successfully!")


M7: ORBITAL-ONLY BASELINE
Model trained successfully!

ORBITAL-ONLY RESULTS


,Method,Evaluation_Subset,Records,PHA_Prevalence,Accuracy,Precision,Recall,F1,ROC_AUC,Average_Precision,TN,FP,FN,TP
0,M7_Orbital_Only,Full_Test_Set,8536,0.0597,0.7528,0.1516,0.6824,0.2480,0.7970,0.1939,6078,1948,162,348
1,M7_Orbital_Only,Measured_Diameter,247,0.2996,0.6437,0.4375,0.6622,0.5269,0.7034,0.4594,110,63,25,49
2,M7_Orbital_Only,Missing_Diameter,8289,0.0526,0.7561,0.1369,0.6858,0.2282,0.7997,0.1769,5968,1885,137,299



Prediction records: 8536
Features used: ['a', 'e', 'i', 'om', 'w']

M7 evaluation completed successfully!


In [3]:
# ============================================================
# H-CONDITIONAL MISSINGNESS ANALYSIS
# Research: ML-Based Classification of Potentially Hazardous NEOs
# ============================================================

import pandas as pd
import numpy as np
import os

# 1. Load the original dataset
url = "https://raw.githubusercontent.com/meherhasija/neo-missingness-lab/refs/heads/main/data/neo_asteroid_dataset.csv"

df_h = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Original shape:", df_h.shape)

# 2. Keep records with PHA label and H value available
df_h = df_h.dropna(subset=["pha", "H"]).copy()

# Standardize PHA labels
df_h["pha"] = df_h["pha"].astype(str).str.strip().str.upper()

# 3. Identify whether diameter is available
df_h["diameter_status"] = np.where(
    df_h["diameter"].notna(),
    "Diameter Available",
    "Diameter Missing"
)

# 4. Create H magnitude groups
bins = [-np.inf, 18, 20, 22, np.inf]
labels = ["H <= 18", "18 < H <= 20", "20 < H <= 22", "H > 22"]

df_h["H_group"] = pd.cut(
    df_h["H"],
    bins=bins,
    labels=labels
)

# 5. Analyse diameter availability across H groups
availability = (
    df_h.groupby("H_group", observed=False)
    .agg(
        Total_Asteroids=("pha", "count"),
        Diameter_Available=("diameter", "count")
    )
    .reset_index()
)

availability["Diameter_Missing"] = (
    availability["Total_Asteroids"] -
    availability["Diameter_Available"]
)

availability["Diameter_Available_Percent"] = (
    availability["Diameter_Available"] /
    availability["Total_Asteroids"] * 100
)

availability["Diameter_Missing_Percent"] = (
    availability["Diameter_Missing"] /
    availability["Total_Asteroids"] * 100
)

print("\n========== DIAMETER AVAILABILITY BY H GROUP ==========")
display(availability.round(2))

# 6. Compare PHA rates for available and missing diameter
pha_analysis = (
    df_h.groupby(
        ["H_group", "diameter_status"],
        observed=False
    )
    .agg(
        Total_Records=("pha", "count"),
        PHA_Count=("pha", lambda x: (x == "Y").sum())
    )
    .reset_index()
)

pha_analysis["PHA_Rate_Percent"] = (
    pha_analysis["PHA_Count"] /
    pha_analysis["Total_Records"] * 100
)

print("\n========== PHA RATE BY H GROUP AND DIAMETER STATUS ==========")
display(pha_analysis.round(2))

# 7. Compare H <= 22 and H > 22
df_h["H_category"] = np.where(
    df_h["H"] <= 22,
    "H <= 22",
    "H > 22"
)

h_summary = (
    df_h.groupby(
        ["H_category", "diameter_status"]
    )
    .agg(
        Total_Records=("pha", "count"),
        PHA_Count=("pha", lambda x: (x == "Y").sum())
    )
    .reset_index()
)

h_summary["PHA_Rate_Percent"] = (
    h_summary["PHA_Count"] /
    h_summary["Total_Records"] * 100
)

print("\n========== PHA RATE: H <= 22 VS H > 22 ==========")
display(h_summary.round(2))

# 8. Investigate PHA-labelled records with H > 22
unexpected_pha = df_h[
    (df_h["pha"] == "Y") &
    (df_h["H"] > 22)
].copy()

print("\n========== PHA = Y WITH H > 22 ==========")
print("Number of records:", len(unexpected_pha))

columns_to_show = [
    col for col in
    ["spkid", "pdes", "full_name", "H", "diameter", "moid", "pha"]
    if col in unexpected_pha.columns
]

display(unexpected_pha[columns_to_show].head(20))

# 9. Save all analysis outputs
output_dir = "/content/neo_evaluation_results"
os.makedirs(output_dir, exist_ok=True)

availability.to_csv(
    os.path.join(output_dir, "h_group_diameter_availability.csv"),
    index=False
)

pha_analysis.to_csv(
    os.path.join(output_dir, "h_group_pha_rates.csv"),
    index=False
)

h_summary.to_csv(
    os.path.join(output_dir, "h_category_pha_rates.csv"),
    index=False
)

unexpected_pha.to_csv(
    os.path.join(output_dir, "pha_records_with_H_above_22.csv"),
    index=False
)

print("\nAnalysis completed successfully!")
print("Files saved in:", output_dir)

Dataset loaded successfully!
Original shape: (42815, 21)

========== DIAMETER AVAILABILITY BY H GROUP ==========


,H_group,Total_Asteroids,Diameter_Available,Diameter_Missing,Diameter_Available_Percent,Diameter_Missing_Percent
0,H <= 18,1092,392,700,35.90,64.10
1,18 < H <= 20,3765,435,3330,11.55,88.45
2,20 < H <= 22,6889,274,6615,3.98,96.02
3,H > 22,30932,143,30789,0.46,99.54



========== PHA RATE BY H GROUP AND DIAMETER STATUS ==========


,H_group,diameter_status,Total_Records,PHA_Count,PHA_Rate_Percent
0,H <= 18,Diameter Available,392,81,20.66
1,H <= 18,Diameter Missing,700,106,15.14
2,18 < H <= 20,Diameter Available,435,130,29.89
3,18 < H <= 20,Diameter Missing,3330,587,17.63
4,20 < H <= 22,Diameter Available,274,127,46.35
5,20 < H <= 22,Diameter Missing,6615,1486,22.46
6,H > 22,Diameter Available,143,2,1.40
7,H > 22,Diameter Missing,30789,31,0.10



========== PHA RATE: H <= 22 VS H > 22 ==========


,H_category,diameter_status,Total_Records,PHA_Count,PHA_Rate_Percent
0,H <= 22,Diameter Available,1101,338,30.70
1,H <= 22,Diameter Missing,10645,2179,20.47
2,H > 22,Diameter Available,143,2,1.40
3,H > 22,Diameter Missing,30789,31,0.10



========== PHA = Y WITH H > 22 ==========
Number of records: 33


,spkid,pdes,full_name,H,diameter,moid,pha
605,20154019,154019,154019 (2002 CZ9),22.07,NaN,0.02560,Y
2189,20468468,468468,468468 (2004 KH17),22.01,0.197,0.00344,Y
2624,20513312,513312,513312 (2007 DM41),22.06,NaN,0.00746,Y
4051,50092240,1999 FR19,(1999 FR19),22.49,NaN,0.03480,Y
4056,50020999,1999 JZ10,(1999 JZ10),22.22,NaN,0.03790,Y
4109,50025789,2000 AA6,(2000 AA6),22.44,0.316,0.00771,Y
4179,50066102,2000 SL10,(2000 SL10),22.16,NaN,0.01090,Y
4202,50063081,2000 WC1,(2000 WC1),22.57,NaN,0.00551,Y
4204,50092299,2000 WH10,(2000 WH10),22.42,NaN,0.04900,Y
4409,50102706,2001 XX4,(2001 XX4),22.10,NaN,0.01130,Y



Analysis completed successfully!
Files saved in: /content/neo_evaluation_results


In [4]:
# ============================================================
# DISCOVERY ERA ANALYSIS
# Check diameter availability across discovery decades
# ============================================================

import pandas as pd
import numpy as np
import os

# 1. Load original dataset
url = "https://raw.githubusercontent.com/meherhasija/neo-missingness-lab/refs/heads/main/data/neo_asteroid_dataset.csv"

df_era = pd.read_csv(url)

print("Dataset loaded:", df_era.shape)

# 2. Check whether first_obs column exists
if "first_obs" not in df_era.columns:
    print("ERROR: first_obs column not found.")
    print("Available columns:", df_era.columns.tolist())

else:
    # 3. Convert first observation date into datetime
    df_era["first_obs_date"] = pd.to_datetime(
        df_era["first_obs"],
        errors="coerce"
    )

    # Count records without usable first observation date
    print("Records with missing/invalid first_obs:",
          df_era["first_obs_date"].isna().sum())

    # 4. Keep records with valid date and PHA label
    df_era = df_era.dropna(
        subset=["first_obs_date", "pha"]
    ).copy()

    df_era["pha"] = (
        df_era["pha"].astype(str).str.strip().str.upper()
    )

    # 5. Extract discovery year and decade
    df_era["discovery_year"] = (
        df_era["first_obs_date"].dt.year
    )

    df_era["discovery_decade"] = (
        (df_era["discovery_year"] // 10) * 10
    )

    # 6. Create diameter availability status
    df_era["diameter_status"] = np.where(
        df_era["diameter"].notna(),
        "Diameter Available",
        "Diameter Missing"
    )

    # 7. Diameter availability by decade
    era_availability = (
        df_era.groupby("discovery_decade")
        .agg(
            Total_Asteroids=("pha", "count"),
            Diameter_Available=("diameter", "count")
        )
        .reset_index()
    )

    era_availability["Diameter_Missing"] = (
        era_availability["Total_Asteroids"]
        - era_availability["Diameter_Available"]
    )

    era_availability["Diameter_Available_Percent"] = (
        era_availability["Diameter_Available"]
        / era_availability["Total_Asteroids"] * 100
    )

    era_availability["Diameter_Missing_Percent"] = (
        era_availability["Diameter_Missing"]
        / era_availability["Total_Asteroids"] * 100
    )

    print("\n========== DIAMETER AVAILABILITY BY DECADE ==========")
    display(era_availability.round(2))

    # 8. PHA rate by decade and diameter status
    era_pha = (
        df_era.groupby(
            ["discovery_decade", "diameter_status"]
        )
        .agg(
            Total_Records=("pha", "count"),
            PHA_Count=("pha", lambda x: (x == "Y").sum())
        )
        .reset_index()
    )

    era_pha["PHA_Rate_Percent"] = (
        era_pha["PHA_Count"]
        / era_pha["Total_Records"] * 100
    )

    print("\n========== PHA RATE BY DECADE AND DIAMETER STATUS ==========")
    display(era_pha.round(2))

    # 9. Save results
    output_dir = "/content/neo_evaluation_results"
    os.makedirs(output_dir, exist_ok=True)

    era_availability.to_csv(
        os.path.join(output_dir, "discovery_era_diameter_availability.csv"),
        index=False
    )

    era_pha.to_csv(
        os.path.join(output_dir, "discovery_era_pha_rates.csv"),
        index=False
    )

    print("\nAnalysis completed successfully!")
    print("Files saved in:", output_dir)

Dataset loaded: (42815, 21)
Records with missing/invalid first_obs: 1

========== DIAMETER AVAILABILITY BY DECADE ==========


,discovery_decade,Total_Asteroids,Diameter_Available,Diameter_Missing,Diameter_Available_Percent,Diameter_Missing_Percent
0,1890,1,1,0,100.00,0.00
1,1910,2,1,1,50.00,50.00
2,1920,2,2,0,100.00,0.00
3,1930,9,5,4,55.56,44.44
4,1940,4,4,0,100.00,0.00
5,1950,82,45,37,54.88,45.12
6,1960,10,6,4,60.00,40.00
7,1970,118,55,63,46.61,53.39
8,1980,229,98,131,42.79,57.21
9,1990,888,160,728,18.02,81.98



========== PHA RATE BY DECADE AND DIAMETER STATUS ==========


,discovery_decade,diameter_status,Total_Records,PHA_Count,PHA_Rate_Percent
0,1890,Diameter Available,1,0,0.00
1,1910,Diameter Available,1,0,0.00
2,1910,Diameter Missing,1,0,0.00
3,1920,Diameter Available,2,0,0.00
4,1930,Diameter Available,5,4,80.00
5,1930,Diameter Missing,4,2,50.00
6,1940,Diameter Available,4,2,50.00
7,1950,Diameter Available,45,13,28.89
8,1950,Diameter Missing,37,6,16.22
9,1960,Diameter Available,6,1,16.67



Analysis completed successfully!
Files saved in: /content/neo_evaluation_results


In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Load the dataset
url = "https://raw.githubusercontent.com/meherhasija/neo-missingness-lab/refs/heads/main/data/neo_asteroid_dataset.csv"
df = pd.read_csv(url)

# Clean relevant columns
df['pha'] = df['pha'].astype(str).str.strip().str.upper()
df['moid'] = pd.to_numeric(df['moid'], errors='coerce')

df = df[df['pha'].isin(['Y', 'N'])].copy()
df = df.dropna(subset=['moid'])

# Actual PHA labels
actual = (df['pha'] == 'Y').astype(int)

# MOID-only rule: MOID <= 0.05 AU
predicted = (df['moid'] <= 0.05).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    actual, predicted, labels=[0, 1]
).ravel()

# Summary metrics
results = pd.DataFrame([{
    'Records': len(df),
    'MOID_Threshold_AU': 0.05,
    'Accuracy': accuracy_score(actual, predicted),
    'Precision': precision_score(actual, predicted, zero_division=0),
    'Recall': recall_score(actual, predicted, zero_division=0),
    'F1': f1_score(actual, predicted, zero_division=0),
    'TN': tn,
    'FP': fp,
    'FN': fn,
    'TP': tp
}])

# Identify disagreements
df['actual_PHA'] = actual
df['MOID_rule_prediction'] = predicted

disagreements = df[
    df['actual_PHA'] != df['MOID_rule_prediction']
].copy()

# Save results
output_dir = '/content/neo_evaluation_results'
os.makedirs(output_dir, exist_ok=True)

results.to_csv(
    os.path.join(output_dir, 'moid_rule_metrics.csv'),
    index=False
)

disagreements.to_csv(
    os.path.join(output_dir, 'moid_rule_disagreements.csv'),
    index=False
)

# Display
print("MOID RULE ANALYSIS")
display(results.round(4))

print("\nConfusion Matrix:")
print([[tn, fp], [fn, tp]])

print("\nDisagreement records:", len(disagreements))
display(disagreements.head(10))

print("\nFiles saved successfully!")

MOID RULE ANALYSIS


,Records,MOID_Threshold_AU,Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,42680,0.05,0.5172,0.1101,1.0,0.1984,19523,20607,0,2550



Confusion Matrix:
[[np.int64(19523), np.int64(20607)], [np.int64(0), np.int64(2550)]]

Disagreement records: 20607


,spkid,pdes,full_name,kind,neo,pha,e,a,q,i,...,H,diameter,albedo,first_obs,last_obs,data_arc,n_obs_used,condition_code,actual_PHA,MOID_rule_prediction
295,20054509,54509,54509 YORP (2000 PH5),an,Y,N,0.2300,1.0060,0.775,1.60,...,22.66,NaN,NaN,2000-08-03,2005-08-03,1826.0,550,0.0,0,1
909,20209215,209215,209215 (2003 WP25),an,Y,N,0.1213,0.9892,0.869,2.56,...,24.30,NaN,NaN,2002-10-25,2009-03-02,2320.0,59,1.0,0,1
1159,20293726,293726,293726 (2007 RQ17),an,Y,N,0.3691,1.5780,0.996,2.01,...,22.64,NaN,NaN,2007-09-09,2015-09-09,2922.0,136,0.0,0,1
1267,20326354,326354,326354 (2000 SJ344),an,Y,N,0.1747,1.1400,0.941,5.75,...,22.78,NaN,NaN,2000-09-30,2023-05-13,8260.0,70,0.0,0,1
1495,20367943,367943,367943 Duende (2012 DA14),an,Y,N,0.0893,0.9106,0.829,11.61,...,24.19,NaN,NaN,2012-02-23,2013-02-21,364.0,1017,0.0,0,1
1697,20410777,410777,410777 (2009 FD),an,Y,N,0.4930,1.1640,0.590,3.12,...,22.18,0.472,0.010,2009-02-24,2024-04-09,5523.0,516,0.0,0,1
1700,20411165,411165,411165 (2010 DF1),an,Y,N,0.5027,1.5990,0.795,20.09,...,22.09,0.159,0.121,2010-02-17,2022-08-30,4577.0,281,0.0,0,1
1963,20443104,443104,443104 (2013 XK22),an,Y,N,0.2033,1.0460,0.833,6.99,...,24.16,NaN,NaN,2013-12-14,2016-07-30,959.0,99,1.0,0,1
1980,20445830,445830,445830 (2012 CL19),an,Y,N,0.4410,1.5610,0.873,2.87,...,22.23,NaN,NaN,2012-02-11,2015-10-03,1330.0,111,0.0,0,1
2014,20450237,450237,450237 (2002 XY38),an,Y,N,0.2177,0.9114,0.713,2.12,...,23.18,0.096,0.134,2002-12-07,2023-01-13,7342.0,145,0.0,0,1



Files saved successfully!


In [6]:

import pandas as pd
import requests
import time
import os

# Load disagreement records
file_path = '/content/neo_evaluation_results/moid_rule_disagreements.csv'
disagreements = pd.read_csv(file_path)

# Select the 17 unusual records:
# MOID <= 0.05 and H <= 22, but actual PHA = N
unusual = disagreements[
    (pd.to_numeric(disagreements['H'], errors='coerce') <= 22) &
    (pd.to_numeric(disagreements['moid'], errors='coerce') <= 0.05) &
    (disagreements['pha'].astype(str).str.upper() == 'N')
].copy()

print("Unusual records selected:", len(unusual))

# Query current JPL SBDB API
results = []

for _, row in unusual.iterrows():

    spkid = str(row['spkid'])

    try:
        response = requests.get(
            'https://ssd-api.jpl.nasa.gov/sbdb.api',
            params={
                'spk': spkid,
                'phys-par': '1',
                'full-prec': '1'
            },
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        obj = data.get('object', {})
        orbit = data.get('orbit', {})

        # Extract current H value
        current_H = None
        for item in data.get('phys_par', []):
            if item.get('name') == 'H':
                current_H = item.get('value')
                break

        results.append({
            'spkid': spkid,
            'pdes': row['pdes'],
            'full_name': row['full_name'],
            'original_PHA': row['pha'],
            'original_MOID': row['moid'],
            'original_H': row['H'],
            'current_JPL_PHA': obj.get('pha'),
            'current_JPL_MOID': orbit.get('moid'),
            'current_JPL_H': current_H,
            'current_orbit_id': orbit.get('orbit_id'),
            'current_solution_date': orbit.get('soln_date'),
            'API_status': 'Success'
        })

    except Exception as e:
        results.append({
            'spkid': spkid,
            'pdes': row['pdes'],
            'full_name': row['full_name'],
            'original_PHA': row['pha'],
            'original_MOID': row['moid'],
            'original_H': row['H'],
            'API_status': str(e)
        })

    time.sleep(0.3)

# Save comparison
jpl_check = pd.DataFrame(results)

output_dir = '/content/neo_evaluation_results'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    'moid_17_records_jpl_verification.csv'
)

jpl_check.to_csv(output_path, index=False)

print("\nJPL VERIFICATION RESULTS")
display(jpl_check)

print("\nSuccessfully queried:",
      (jpl_check['API_status'] == 'Success').sum(),
      "out of", len(jpl_check))

print("\nSaved:", output_path)

Unusual records selected: 17

JPL VERIFICATION RESULTS


,spkid,pdes,full_name,original_PHA,original_MOID,original_H,current_JPL_PHA,current_JPL_MOID,current_JPL_H,current_orbit_id,current_solution_date,API_status
0,20853358,853358,853358 (2009 DL46),N,0.012300,21.99,False,.012268,21.99,77,2025-10-30 18:15:19,Success
1,20886097,886097,886097 (2019 VO1),N,0.007470,22.00,False,.00747051,22.00,25,2026-03-14 17:41:59,Success
2,50342338,2006 QJ65,(2006 QJ65),N,0.047000,22.00,False,.0469711,22.0,9,2021-04-14 23:46:28,Success
3,50441137,2008 XM1,(2008 XM1),N,0.003140,22.00,False,.00313519,22.0,18,2021-04-15 02:22:12,Success
4,50465742,2009 QS,(2009 QS),N,0.042200,22.00,False,.0422282,22.0,21,2021-04-15 02:48:39,Success
5,50604506,2012 KN11,(2012 KN11),N,0.020500,21.90,False,.0204949,21.9,13,2021-04-15 06:26:18,Success
6,50640719,2013 LF7,(2013 LF7),N,0.008480,22.00,False,.00847847,22.0,24,2021-04-15 08:14:03,Success
7,50652879,2013 UX,(2013 UX),N,0.035900,22.00,False,.0358542,22.0,14,2021-04-15 08:52:32,Success
8,50661523,2014 CM13,(2014 CM13),N,0.021000,22.00,False,.0209939,22.0,20,2021-04-15 09:20:26,Success
9,50703813,2015 AM45,(2015 AM45),N,0.000487,22.00,False,.000487491,22.0,17,2021-04-15 12:17:36,Success



Successfully queried: 17 out of 17

Saved: /content/neo_evaluation_results/moid_17_records_jpl_verification.csv


In [7]:
# ============================================================
# REPEATED STRATIFIED EVALUATION: M1-M7
# 10 SPLITS + BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

import os
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DATA_URL = (
    "https://raw.githubusercontent.com/meherhasija/"
    "neo-missingness-lab/refs/heads/main/"
    "data/neo_asteroid_dataset.csv"
)

FEATURES = ['a', 'e', 'i', 'om', 'w', 'diameter']
ORBITAL_FEATURES = ['a', 'e', 'i', 'om', 'w']

N_SPLITS = 10
TEST_SIZE = 0.20
BOOTSTRAP_ITERATIONS = 200
BASE_SEED = 42

OUTPUT_DIR = '/content/neo_evaluation_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 2. LOAD AND PREPARE DATA
# ------------------------------------------------------------

df_repeat = pd.read_csv(DATA_URL)

df_repeat = df_repeat.dropna(subset=['pha']).copy()
df_repeat['pha'] = (
    df_repeat['pha'].astype(str).str.strip().str.upper()
)
df_repeat = df_repeat[df_repeat['pha'].isin(['Y', 'N'])].copy()

df_repeat['target'] = df_repeat['pha'].map({'N': 0, 'Y': 1})

X_repeat = df_repeat[FEATURES].copy()
y_repeat = df_repeat['target'].copy()

print("Dataset records:", len(df_repeat))
print("PHA-positive records:", int(y_repeat.sum()))
print("PHA prevalence:", round(y_repeat.mean(), 4))

# ------------------------------------------------------------
# 3. MODEL AND METRIC FUNCTIONS
# ------------------------------------------------------------

def make_repeat_model(seed):
    return HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        random_state=seed,
        class_weight='balanced',
        early_stopping=False
    )

def get_metrics(y_true, probabilities):
    predictions = (probabilities >= 0.5).astype(int)

    return {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(
            y_true, predictions, zero_division=0
        ),
        'Recall': recall_score(
            y_true, predictions, zero_division=0
        ),
        'F1': f1_score(
            y_true, predictions, zero_division=0
        ),
        'ROC_AUC': roc_auc_score(y_true, probabilities),
        'Average_Precision': average_precision_score(
            y_true, probabilities
        )
    }

# ------------------------------------------------------------
# 4A. BOOTSTRAP METRICS
# ------------------------------------------------------------

def bootstrap_metrics(y_true, probabilities, seed):
    rng = np.random.default_rng(seed)

    y_array = np.asarray(y_true)
    p_array = np.asarray(probabilities)

    boot_values = {
        'F1': [],
        'Average_Precision': [],
        'ROC_AUC': []
    }

    n = len(y_array)

    for _ in range(BOOTSTRAP_ITERATIONS):
        idx = rng.integers(0, n, size=n)

        y_sample = y_array[idx]
        p_sample = p_array[idx]

        # Skip samples containing only one class
        if len(np.unique(y_sample)) < 2:
            continue

        pred_sample = (p_sample >= 0.5).astype(int)

        boot_values['F1'].append(
            f1_score(y_sample, pred_sample, zero_division=0)
        )
        boot_values['Average_Precision'].append(
            average_precision_score(y_sample, p_sample)
        )
        boot_values['ROC_AUC'].append(
            roc_auc_score(y_sample, p_sample)
        )

    return boot_values

# ------------------------------------------------------------
# 4B. PAIRED STRATIFIED BOOTSTRAP DIFFERENCES
# ------------------------------------------------------------

PAIRED_BOOTSTRAP_ITERATIONS = 1000

def paired_bootstrap_comparison(
    y_true, reference_probs, comparison_probs, seed
):
    rng = np.random.default_rng(seed)

    y = np.asarray(y_true)
    ref = np.asarray(reference_probs)
    comp = np.asarray(comparison_probs)

    negative_idx = np.where(y == 0)[0]
    positive_idx = np.where(y == 1)[0]

    differences = {
        'F1_Difference': [],
        'Average_Precision_Difference': [],
        'ROC_AUC_Difference': []
    }

    for _ in range(PAIRED_BOOTSTRAP_ITERATIONS):

        # Stratified sampling keeps both classes represented
        sampled_neg = rng.choice(
            negative_idx, size=len(negative_idx), replace=True
        )
        sampled_pos = rng.choice(
            positive_idx, size=len(positive_idx), replace=True
        )

        idx = np.concatenate([sampled_neg, sampled_pos])
        rng.shuffle(idx)

        ys = y[idx]
        rp = ref[idx]
        cp = comp[idx]

        ref_pred = (rp >= 0.5).astype(int)
        comp_pred = (cp >= 0.5).astype(int)

        differences['F1_Difference'].append(
            f1_score(ys, comp_pred, zero_division=0)
            - f1_score(ys, ref_pred, zero_division=0)
        )

        differences['Average_Precision_Difference'].append(
            average_precision_score(ys, cp)
            - average_precision_score(ys, rp)
        )

        differences['ROC_AUC_Difference'].append(
            roc_auc_score(ys, cp)
            - roc_auc_score(ys, rp)
        )

    return differences

# ------------------------------------------------------------
# 5. KNN DIAMETER IMPUTATION
# ------------------------------------------------------------

def knn_impute_repeat(X_train, X_test):

    scaler = StandardScaler()

    train_orbit = scaler.fit_transform(
        X_train[ORBITAL_FEATURES]
    )

    test_orbit = scaler.transform(
        X_test[ORBITAL_FEATURES]
    )

    donor_mask = X_train['diameter'].notna()

    donor_orbit = train_orbit[donor_mask.values]
    donor_diameter = X_train.loc[
        donor_mask, 'diameter'
    ].values

    if len(donor_diameter) < 5:
        raise ValueError("Insufficient KNN donor records.")

    knn = NearestNeighbors(n_neighbors=5)
    knn.fit(donor_orbit)

    def fill(data, scaled_orbit):
        result = data.copy()
        missing = result['diameter'].isna()

        if missing.any():
            distances, indices = knn.kneighbors(
                scaled_orbit[missing.values]
            )

            result.loc[missing, 'diameter'] = (
                donor_diameter[indices].mean(axis=1)
            )

        return result

    return (
        fill(X_train, train_orbit),
        fill(X_test, test_orbit)
    )

# ------------------------------------------------------------
# 6. REPEATED STRATIFIED SPLITS
# ------------------------------------------------------------

splitter = StratifiedShuffleSplit(
    n_splits=N_SPLITS,
    test_size=TEST_SIZE,
    random_state=BASE_SEED
)

repeat_results = []
bootstrap_results = []
prediction_records = []
paired_difference_results = []

start_time = time.time()

for split_number, (train_idx, test_idx) in enumerate(
    splitter.split(X_repeat, y_repeat), start=1
):

    print(f"\nRunning split {split_number}/{N_SPLITS}")

    X_train_r = X_repeat.iloc[train_idx].copy()
    X_test_r = X_repeat.iloc[test_idx].copy()

    y_train_r = y_repeat.iloc[train_idx].copy()
    y_test_r = y_repeat.iloc[test_idx].copy()

    # Define test subsets
    subsets = {
        'Full_Test_Set': np.ones(len(X_test_r), dtype=bool),
        'Measured_Diameter': X_test_r['diameter'].notna().values,
        'Missing_Diameter': X_test_r['diameter'].isna().values
    }

    # Store method predictions
    method_probabilities = {}

    # --------------------------------------------------------
    # M1: NATIVE MISSING-VALUE HANDLING
    # --------------------------------------------------------

    model = make_repeat_model(split_number)
    model.fit(X_train_r, y_train_r)

    method_probabilities['M1_Native'] = (
        model.predict_proba(X_test_r)[:, 1]
    )

    # --------------------------------------------------------
    # M2: MEDIAN IMPUTATION
    # --------------------------------------------------------

    imputer = SimpleImputer(strategy='median')

    train_data = imputer.fit_transform(X_train_r)
    test_data = imputer.transform(X_test_r)

    model = make_repeat_model(split_number)
    model.fit(train_data, y_train_r)

    method_probabilities['M2_Median'] = (
        model.predict_proba(test_data)[:, 1]
    )

    # --------------------------------------------------------
    # M3: MEDIAN + MISSINGNESS INDICATOR
    # --------------------------------------------------------

    train_m3 = X_train_r.copy()
    test_m3 = X_test_r.copy()

    train_m3['diameter_missing'] = (
        train_m3['diameter'].isna().astype(int)
    )
    test_m3['diameter_missing'] = (
        test_m3['diameter'].isna().astype(int)
    )

    m3_imputer = SimpleImputer(strategy='median')

    train_m3_data = m3_imputer.fit_transform(train_m3)
    test_m3_data = m3_imputer.transform(test_m3)

    model = make_repeat_model(split_number)
    model.fit(train_m3_data, y_train_r)

    method_probabilities['M3_Median_Indicator'] = (
        model.predict_proba(test_m3_data)[:, 1]
    )

    # --------------------------------------------------------
    # M4: KNN IMPUTATION
    # --------------------------------------------------------

    train_knn, test_knn = knn_impute_repeat(
        X_train_r, X_test_r
    )

    model = make_repeat_model(split_number)
    model.fit(train_knn, y_train_r)

    method_probabilities['M4_KNN'] = (
        model.predict_proba(test_knn)[:, 1]
    )

    # --------------------------------------------------------
    # M5: ITERATIVE IMPUTATION
    # --------------------------------------------------------

    iterative_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('imputer', IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=10,
            random_state=42,
            skip_complete=True
        ))
    ])

    train_iter = iterative_pipeline.fit_transform(X_train_r)
    test_iter = iterative_pipeline.transform(X_test_r)

    model = make_repeat_model(split_number)
    model.fit(train_iter, y_train_r)

    method_probabilities['M5_Iterative'] = (
        model.predict_proba(test_iter)[:, 1]
    )

    # --------------------------------------------------------
    # M6: COMPLETE-CASE ANALYSIS
    # --------------------------------------------------------

    train_complete = X_train_r.notna().all(axis=1)
    test_complete = X_test_r.notna().all(axis=1)

    X_train_complete = X_train_r.loc[train_complete]
    y_train_complete = y_train_r.loc[train_complete]

    X_test_complete = X_test_r.loc[test_complete]
    y_test_complete = y_test_r.loc[test_complete]

    if (
        len(X_train_complete) > 0
        and len(X_test_complete) > 0
        and y_train_complete.nunique() == 2
        and y_test_complete.nunique() == 2
    ):
        model = make_repeat_model(split_number)
        model.fit(X_train_complete, y_train_complete)

        method_probabilities['M6_Complete_Case'] = (
            model.predict_proba(X_test_complete)[:, 1]
        )

    # --------------------------------------------------------
    # M7: ORBITAL-ONLY BASELINE
    # --------------------------------------------------------

    model = make_repeat_model(split_number)

    model.fit(
        X_train_r[ORBITAL_FEATURES],
        y_train_r
    )

    method_probabilities['M7_Orbital_Only'] = (
        model.predict_proba(
            X_test_r[ORBITAL_FEATURES]
        )[:, 1]
    )

# ------------------------------------------------------------
    # 7. SAVE PREDICTIONS AND RUN PAIRED COMPARISONS
    # ------------------------------------------------------------

    # Save predictions for all methods on the same test records.
    for method_name, probabilities in method_probabilities.items():

        if method_name == 'M6_Complete_Case':
            positions = np.where(test_complete.values)[0]
            actual = y_test_complete.values
            probs = probabilities
        else:
            positions = np.arange(len(X_test_r))
            actual = y_test_r.values
            probs = probabilities

        for j, position in enumerate(positions):
            prediction_records.append({
                'Split': split_number,
                'Test_Row_Index': int(test_idx[position]),
                'Method': method_name,
                'Actual_PHA': int(actual[j]),
                'Predicted_Probability': float(probs[j])
            })

    # Evaluate M1-M5 and M7 on common test subsets.
    for method_name, probabilities in method_probabilities.items():

        if method_name == 'M6_Complete_Case':
            continue

        for subset_name, mask in subsets.items():

            if mask.sum() == 0:
                continue

            actual = y_test_r.values[mask]
            probs = probabilities[mask]

            if len(np.unique(actual)) < 2:
                continue

            metrics = get_metrics(actual, probs)

            repeat_results.append({
                'Split': split_number,
                'Method': method_name,
                'Evaluation_Subset': subset_name,
                'Records': len(actual),
                **metrics
            })

            boot = bootstrap_metrics(
                actual, probs, seed=BASE_SEED + split_number
            )

            for metric_name, values in boot.items():
                if len(values) > 0:
                    lower, upper = np.percentile(
                        values, [2.5, 97.5]
                    )

                    bootstrap_results.append({
                        'Split': split_number,
                        'Method': method_name,
                        'Evaluation_Subset': subset_name,
                        'Metric': metric_name,
                        'Bootstrap_Lower_95': lower,
                        'Bootstrap_Upper_95': upper
                    })

    # Paired comparisons: each method minus M1.
    # M6 is compared only on complete-case test records.
    comparison_methods = [
        'M2_Median',
        'M3_Median_Indicator',
        'M4_KNN',
        'M5_Iterative',
        'M7_Orbital_Only'
    ]

    for subset_name, mask in subsets.items():

        actual = y_test_r.values[mask]

        if len(np.unique(actual)) < 2:
            continue

        reference_probs = method_probabilities['M1_Native'][mask]

        for method_name in comparison_methods:

            comparison_probs = method_probabilities[method_name][mask]

            differences = paired_bootstrap_comparison(
                actual,
                reference_probs,
                comparison_probs,
                seed=BASE_SEED + split_number
            )

            for metric_name, values in differences.items():

                lower, upper = np.percentile(values, [2.5, 97.5])

                paired_difference_results.append({
                    'Split': split_number,
                    'Reference_Method': 'M1_Native',
                    'Comparison_Method': method_name,
                    'Evaluation_Subset': subset_name,
                    'Metric': metric_name,
                    'Mean_Difference': np.mean(values),
                    'Bootstrap_Lower_95': lower,
                    'Bootstrap_Upper_95': upper
                })

    # M6: compare against M1 on exactly the same complete-case records.
    if (
        len(X_test_complete) > 0
        and y_test_complete.nunique() == 2
        and 'M6_Complete_Case' in method_probabilities
    ):

        complete_mask = test_complete.values

        actual = y_test_complete.values
        m1_probs = method_probabilities['M1_Native'][complete_mask]
        m6_probs = method_probabilities['M6_Complete_Case']

        m6_metrics = get_metrics(actual, m6_probs)

        repeat_results.append({
            'Split': split_number,
            'Method': 'M6_Complete_Case',
            'Evaluation_Subset': 'Complete_Case',
            'Records': len(actual),
            **m6_metrics
        })

        differences = paired_bootstrap_comparison(
            actual,
            m1_probs,
            m6_probs,
            seed=BASE_SEED + split_number
        )

        for metric_name, values in differences.items():

            lower, upper = np.percentile(values, [2.5, 97.5])

            paired_difference_results.append({
                'Split': split_number,
                'Reference_Method': 'M1_Native',
                'Comparison_Method': 'M6_Complete_Case',
                'Evaluation_Subset': 'Complete_Case',
                'Metric': metric_name,
                'Mean_Difference': np.mean(values),
                'Bootstrap_Lower_95': lower,
                'Bootstrap_Upper_95': upper
            })

    # ------------------------------------------------------------
# 8. SAVE REPEATED SPLIT RESULTS
# ------------------------------------------------------------

repeated_df = pd.DataFrame(repeat_results)
bootstrap_df = pd.DataFrame(bootstrap_results)

predictions_df = pd.DataFrame(prediction_records)
paired_df = pd.DataFrame(paired_difference_results)

predictions_df.to_csv(
    os.path.join(OUTPUT_DIR, 'repeated_split_predictions.csv'),
    index=False
)

paired_df.to_csv(
    os.path.join(OUTPUT_DIR, 'paired_bootstrap_differences.csv'),
    index=False
)

paired_summary = (
    paired_df
    .groupby([
        'Comparison_Method',
        'Evaluation_Subset',
        'Metric'
    ])
    .agg(
        Mean_Difference=('Mean_Difference', 'mean'),
        Mean_CI_Lower=('Bootstrap_Lower_95', 'mean'),
        Mean_CI_Upper=('Bootstrap_Upper_95', 'mean')
    )
    .reset_index()
)

paired_summary.to_csv(
    os.path.join(OUTPUT_DIR, 'paired_bootstrap_summary.csv'),
    index=False
)

repeated_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'repeated_10_split_metrics.csv'
    ),
    index=False
)

bootstrap_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'repeated_bootstrap_confidence_intervals.csv'
    ),
    index=False
)


# ------------------------------------------------------------
# 9. SUMMARY: MEAN AND STANDARD DEVIATION
# ------------------------------------------------------------

metric_columns = [
    'Accuracy', 'Precision', 'Recall',
    'F1', 'ROC_AUC', 'Average_Precision'
]

summary_df = (
    repeated_df
    .groupby(['Method', 'Evaluation_Subset'])[metric_columns]
    .agg(['mean', 'std'])
    .reset_index()
)

summary_df.columns = [
    '_'.join(str(x) for x in col if x != '')
    if isinstance(col, tuple) else col
    for col in summary_df.columns
]

summary_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'repeated_evaluation_summary.csv'
    ),
    index=False
)

# ------------------------------------------------------------
# 10. SUMMARIZE BOOTSTRAP INTERVALS
# ------------------------------------------------------------

bootstrap_summary = (
    bootstrap_df
    .groupby(['Method', 'Evaluation_Subset', 'Metric'])
    .agg(
        Mean_Bootstrap_Lower=('Bootstrap_Lower_95', 'mean'),
        Mean_Bootstrap_Upper=('Bootstrap_Upper_95', 'mean')
    )
    .reset_index()
)

bootstrap_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        'repeated_bootstrap_summary.csv'
    ),
    index=False
)

# ------------------------------------------------------------
# 11. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("REPEATED EVALUATION SUMMARY")
print("=" * 65)

display(summary_df.round(4))

print("\nBOOTSTRAP CONFIDENCE INTERVAL SUMMARY")
display(bootstrap_summary.round(4))

print("\nTotal evaluation rows:", len(repeated_df))
print("Total bootstrap interval rows:", len(bootstrap_df))
print("Time taken (minutes):", round((time.time() - start_time) / 60, 2))

print("\nFILES SAVED:")
print("1. repeated_10_split_metrics.csv")
print("2. repeated_evaluation_summary.csv")
print("3. repeated_bootstrap_confidence_intervals.csv")
print("4. repeated_bootstrap_summary.csv")
print("5. repeated_split_predictions.csv")
print("6. paired_bootstrap_differences.csv")
print("7. paired_bootstrap_summary.csv")

print("\nREPEATED EVALUATION COMPLETED!")

Dataset records: 42680
PHA-positive records: 2550
PHA prevalence: 0.0597

Running split 1/10

Running split 2/10

Running split 3/10

Running split 4/10

Running split 5/10

Running split 6/10

Running split 7/10

Running split 8/10

Running split 9/10

Running split 10/10

REPEATED EVALUATION SUMMARY


,Method,Evaluation_Subset,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_mean,F1_std,ROC_AUC_mean,ROC_AUC_std,Average_Precision_mean,Average_Precision_std
0,M1_Native,Full_Test_Set,0.7671,0.0054,0.1629,0.0055,0.7002,0.0305,0.2642,0.0090,0.8199,0.0088,0.2667,0.0139
1,M1_Native,Measured_Diameter,0.6016,0.0280,0.4049,0.0244,0.9318,0.0323,0.5641,0.0251,0.8160,0.0210,0.6291,0.0277
2,M1_Native,Missing_Diameter,0.7721,0.0058,0.1438,0.0054,0.6637,0.0328,0.2364,0.0091,0.8052,0.0089,0.1840,0.0113
3,M2_Median,Full_Test_Set,0.7613,0.0058,0.1579,0.0047,0.6910,0.0253,0.2570,0.0075,0.8158,0.0078,0.2487,0.0124
4,M2_Median,Measured_Diameter,0.5835,0.0135,0.3908,0.0204,0.9056,0.0324,0.5457,0.0220,0.7715,0.0162,0.5722,0.0376
5,M2_Median,Missing_Diameter,0.7667,0.0058,0.1397,0.0050,0.6569,0.0303,0.2304,0.0084,0.8035,0.0096,0.1828,0.0137
6,M3_Median_Indicator,Full_Test_Set,0.7662,0.0045,0.1603,0.0045,0.6875,0.0278,0.2600,0.0076,0.8178,0.0076,0.2648,0.0123
7,M3_Median_Indicator,Measured_Diameter,0.5720,0.0285,0.3868,0.0272,0.9281,0.0329,0.5454,0.0275,0.8105,0.0237,0.6326,0.0361
8,M3_Median_Indicator,Missing_Diameter,0.7721,0.0045,0.1416,0.0054,0.6493,0.0295,0.2325,0.0089,0.8032,0.0086,0.1826,0.0120
9,M4_KNN,Full_Test_Set,0.7486,0.0069,0.1458,0.0035,0.6600,0.0232,0.2388,0.0056,0.7935,0.0067,0.1892,0.0111



BOOTSTRAP CONFIDENCE INTERVAL SUMMARY


,Method,Evaluation_Subset,Metric,Mean_Bootstrap_Lower,Mean_Bootstrap_Upper
0,M1_Native,Full_Test_Set,Average_Precision,0.2308,0.3067
1,M1_Native,Full_Test_Set,F1,0.2429,0.2857
2,M1_Native,Full_Test_Set,ROC_AUC,0.8034,0.8364
3,M1_Native,Measured_Diameter,Average_Precision,0.5168,0.7395
4,M1_Native,Measured_Diameter,F1,0.4858,0.6409
5,M1_Native,Measured_Diameter,ROC_AUC,0.7596,0.8665
6,M1_Native,Missing_Diameter,Average_Precision,0.1587,0.2163
7,M1_Native,Missing_Diameter,F1,0.2140,0.2569
8,M1_Native,Missing_Diameter,ROC_AUC,0.7866,0.8215
9,M2_Median,Full_Test_Set,Average_Precision,0.2146,0.2867



Total evaluation rows: 190
Total bootstrap interval rows: 540
Time taken (minutes): 63.97

FILES SAVED:
1. repeated_10_split_metrics.csv
2. repeated_evaluation_summary.csv
3. repeated_bootstrap_confidence_intervals.csv
4. repeated_bootstrap_summary.csv
5. repeated_split_predictions.csv
6. paired_bootstrap_differences.csv
7. paired_bootstrap_summary.csv

REPEATED EVALUATION COMPLETED!


In [8]:
import os

folder = "/content/neo_evaluation_results"

print("FILES IN OUTPUT FOLDER:\n")

for filename in sorted(os.listdir(folder)):
    path = os.path.join(folder, filename)
    print(filename, "|", os.path.getsize(path), "bytes")

FILES IN OUTPUT FOLDER:

all_methods_metrics.csv | 3034 bytes
all_methods_predictions.csv | 1776917 bytes
discovery_era_diameter_availability.csv | 651 bytes
discovery_era_pha_rates.csv | 1032 bytes
full_test_set_metrics.csv | 1039 bytes
h_category_pha_rates.csv | 281 bytes
h_group_diameter_availability.csv | 356 bytes
h_group_pha_rates.csv | 505 bytes
measured_diameter_metrics.csv | 1169 bytes
missing_diameter_metrics.csv | 1056 bytes
moid_17_records_jpl_verification.csv | 1894 bytes
moid_rule_disagreements.csv | 2739792 bytes
moid_rule_metrics.csv | 158 bytes
orbital_only_metrics.csv | 670 bytes
orbital_only_predictions.csv | 387121 bytes
paired_bootstrap_differences.csv | 60811 bytes
paired_bootstrap_summary.csv | 5581 bytes
pha_records_with_H_above_22.csv | 5554 bytes
repeated_10_split_metrics.csv | 28631 bytes
repeated_bootstrap_confidence_intervals.csv | 42845 bytes
repeated_bootstrap_summary.csv | 4240 bytes
repeated_evaluation_summary.csv | 5295 bytes
repeated_split_predictions

In [9]:
import pandas as pd
import os

folder = "/content/neo_evaluation_results"

files_to_check = [
    "repeated_10_split_metrics.csv",
    "repeated_evaluation_summary.csv",
    "repeated_bootstrap_confidence_intervals.csv",
    "repeated_bootstrap_summary.csv",
    "repeated_split_predictions.csv",
    "paired_bootstrap_differences.csv",
    "paired_bootstrap_summary.csv"
]

for filename in files_to_check:
    path = os.path.join(folder, filename)
    df = pd.read_csv(path)

    print(f"\n{filename}")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    if "Split" in df.columns:
        print("Splits:", sorted(df["Split"].unique()))


repeated_10_split_metrics.csv
Rows: 190
Columns: ['Split', 'Method', 'Evaluation_Subset', 'Records', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'Average_Precision']
Splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

repeated_evaluation_summary.csv
Rows: 19
Columns: ['Method', 'Evaluation_Subset', 'Accuracy_mean', 'Accuracy_std', 'Precision_mean', 'Precision_std', 'Recall_mean', 'Recall_std', 'F1_mean', 'F1_std', 'ROC_AUC_mean', 'ROC_AUC_std', 'Average_Precision_mean', 'Average_Precision_std']

repeated_bootstrap_confidence_intervals.csv
Rows: 540
Columns: ['Split', 'Method', 'Evaluation_Subset', 'Metric', 'Bootstrap_Lower_95', 'Bootstrap_Upper_95']
Splits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

repeated_bootstrap_summary.csv
Rows: 54
Columns: ['Method', 'Evaluation_Subset', 'Metric', 'Mean_Boots

In [10]:
import pandas as pd

# Load the 17-record JPL verification file from GitHub
url = (
    "https://raw.githubusercontent.com/"
    "meherhasija/neo-missingness-lab/main/"
    "results/analysis/moid_17_records_jpl_verification.csv"
)

df_verification = pd.read_csv(url)

print("Total verified records:", len(df_verification))

print("\nAvailable columns:")
print(df_verification.columns.tolist())

display(df_verification)

Total verified records: 17

Available columns:
['spkid', 'pdes', 'full_name', 'original_PHA', 'original_MOID', 'original_H', 'current_JPL_PHA', 'current_JPL_MOID', 'current_JPL_H', 'current_orbit_id', 'current_solution_date', 'API_status']


,spkid,pdes,full_name,original_PHA,original_MOID,original_H,current_JPL_PHA,current_JPL_MOID,current_JPL_H,current_orbit_id,current_solution_date,API_status
0,20853358,853358,853358 (2009 DL46),N,0.012300,21.99,False,0.012268,21.99,77,2025-10-30 18:15:19,Success
1,20886097,886097,886097 (2019 VO1),N,0.007470,22.00,False,0.007471,22.00,25,2026-03-14 17:41:59,Success
2,50342338,2006 QJ65,(2006 QJ65),N,0.047000,22.00,False,0.046971,22.00,9,2021-04-14 23:46:28,Success
3,50441137,2008 XM1,(2008 XM1),N,0.003140,22.00,False,0.003135,22.00,18,2021-04-15 02:22:12,Success
4,50465742,2009 QS,(2009 QS),N,0.042200,22.00,False,0.042228,22.00,21,2021-04-15 02:48:39,Success
5,50604506,2012 KN11,(2012 KN11),N,0.020500,21.90,False,0.020495,21.90,13,2021-04-15 06:26:18,Success
6,50640719,2013 LF7,(2013 LF7),N,0.008480,22.00,False,0.008478,22.00,24,2021-04-15 08:14:03,Success
7,50652879,2013 UX,(2013 UX),N,0.035900,22.00,False,0.035854,22.00,14,2021-04-15 08:52:32,Success
8,50661523,2014 CM13,(2014 CM13),N,0.021000,22.00,False,0.020994,22.00,20,2021-04-15 09:20:26,Success
9,50703813,2015 AM45,(2015 AM45),N,0.000487,22.00,False,0.000487,22.00,17,2021-04-15 12:17:36,Success


### MOID Rule Disagreement Verification

A total of 17 records were identified as disagreements
between the simplified MOID rule and the original PHA labels.

All 17 records were successfully verified against NASA/JPL
Small-Body Database data.

The original dataset classified all 17 records as non-PHAs.
The current JPL PHA flag also returned False for all 17.

However, all 17 objects had an MOID below the 0.05 AU
threshold used by the simplified rule.

This demonstrates that an MOID-only rule can produce
false-positive PHA classifications.

The original PHA labels were retained. No records were
relabelled based on the simplified rule.

Some records have an absolute magnitude of exactly 22.00
despite having a False JPL PHA flag. These boundary cases
require further investigation before a definitive
explanation can be given.

In [11]:

import pandas as pd
import requests
import time
import os
from datetime import datetime, timezone

# 1. Load the 33 records generated by the H-conditional analysis above
input_path = "/content/neo_evaluation_results/pha_records_with_H_above_22.csv"
df = pd.read_csv(input_path)
print("Loaded local input records:", len(df))

if len(df) != 33:
    raise ValueError(f"Expected 33 records, found {len(df)}")

print("Loaded:", len(df), "records")

# 2. Verify each record using JPL SBDB
results = []

for index, row in df.iterrows():

    spkid = str(int(row["spkid"]))

    params = {
        "spk": spkid,
        "full-prec": "true",
        "phys-par": "1"
    }

    try:
        r = requests.get(
            "https://ssd-api.jpl.nasa.gov/sbdb.api",
            params=params,
            timeout=30
        )
        r.raise_for_status()

        data = r.json()

        if "object" not in data:
            raise ValueError("Object not found in API response")

        obj = data["object"]
        orbit = data.get("orbit", {})
        phys = data.get("phys_par", [])

        physical_values = {
            item["name"]: item.get("value")
            for item in phys
        }

        results.append({
            "spkid": spkid,
            "original_name": row["full_name"],
            "original_PHA": row["pha"],
            "original_H": row["H"],
            "original_MOID": row["moid"],
            "JPL_name": obj.get("fullname"),
            "JPL_kind": obj.get("kind"),
            "JPL_NEO": obj.get("neo"),
            "JPL_PHA": obj.get("pha"),  # CORRECTED
            "JPL_H": physical_values.get("H"),
            "JPL_MOID": orbit.get("moid"),
            "JPL_orbit_id": orbit.get("orbit_id"),
            "verification_status": "Success"
        })

        print(f"Verified {index + 1}/33: {row['full_name']}")

    except Exception as e:
        results.append({
            "spkid": spkid,
            "original_name": row["full_name"],
            "verification_status": "Error: " + str(e)
        })

        print(f"Error: {row['full_name']} - {e}")

    time.sleep(0.2)

# 3. Save verification results
result_df = pd.DataFrame(results)

output_dir = "/content/neo_evaluation_results"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    "pha_33_records_jpl_verification.csv"
)

result_df.to_csv(output_file, index=False)

# 4. Display summary
print("\n========== VERIFICATION SUMMARY ==========")
print("Total records:", len(result_df))
print(
    "Successful:",
    (result_df["verification_status"] == "Success").sum()
)
print(
    "Failed:",
    (result_df["verification_status"] != "Success").sum()
)

print("\nJPL PHA classification:")
print(result_df["JPL_PHA"].value_counts(dropna=False))

print("\nFull results:")
display(result_df)

print("\nSaved:", output_file)
print("UTC verification time:", datetime.now(timezone.utc).isoformat())


Loaded local input records: 33
Loaded: 33 records
Verified 1/33: 154019 (2002 CZ9)
Verified 2/33: 468468 (2004 KH17)
Verified 3/33: 513312 (2007 DM41)
Verified 4/33:        (1999 FR19)
Verified 5/33:        (1999 JZ10)
Verified 6/33:        (2000 AA6)
Verified 7/33:        (2000 SL10)
Verified 8/33:        (2000 WC1)
Verified 9/33:        (2000 WH10)
Verified 10/33:        (2001 XX4)
Verified 11/33:        (2001 XP31)
Verified 12/33:        (2002 AS4)
Verified 13/33:        (2002 XO14)
Verified 14/33:        (2004 HG12)
Verified 15/33:        (2004 RE84)
Verified 16/33:        (2007 GQ3)
Verified 17/33:        (2008 AZ30)
Verified 18/33:        (2008 CC6)
Verified 19/33:        (2008 FW5)
Verified 20/33:        (2008 PF1)
Verified 21/33:        (2011 CX46)
Verified 22/33:        (2014 KS40)
Verified 23/33:        (2014 WR367)
Verified 24/33:        (2016 AF193)
Verified 25/33:        (2016 JW5)
Verified 26/33:        (2017 JE3)
Verified 27/33:        (2017 WX12)
Verified 28/33:        

,spkid,original_name,original_PHA,original_H,original_MOID,JPL_name,JPL_kind,JPL_NEO,JPL_PHA,JPL_H,JPL_MOID,JPL_orbit_id,verification_status
0,20154019,154019 (2002 CZ9),Y,22.07,0.02560,154019 (2002 CZ9),an,True,True,22.07,.0256281,15,Success
1,20468468,468468 (2004 KH17),Y,22.01,0.00344,468468 (2004 KH17),an,True,True,22.01,.00344174,64,Success
2,20513312,513312 (2007 DM41),Y,22.06,0.00746,513312 (2007 DM41),an,True,True,22.06,.0074645,51,Success
3,50092240,(1999 FR19),Y,22.49,0.03480,(1999 FR19),au,True,True,22.49,.0348184,18,Success
4,50020999,(1999 JZ10),Y,22.22,0.03790,(1999 JZ10),au,True,True,22.22,.0378895,12,Success
5,50025789,(2000 AA6),Y,22.44,0.00771,(2000 AA6),au,True,True,22.44,.00770636,34,Success
6,50066102,(2000 SL10),Y,22.16,0.01090,(2000 SL10),au,True,True,22.16,.0108875,13,Success
7,50063081,(2000 WC1),Y,22.57,0.00551,(2000 WC1),au,True,True,22.57,.00550594,18,Success
8,50092299,(2000 WH10),Y,22.42,0.04900,(2000 WH10),au,True,True,22.42,.0489846,13,Success
9,50102706,(2001 XX4),Y,22.10,0.01130,(2001 XX4),au,True,True,22.1,.0113228,13,Success



Saved: /content/neo_evaluation_results/pha_33_records_jpl_verification.csv
UTC verification time: 2026-11-04T17:14:11.708875+00:00


In [12]:
import requests
import pandas as pd
import os
import time

# --------------------------------------------------
# 1. Load the 33 records from GitHub
# --------------------------------------------------

input_path = "/content/neo_evaluation_results/pha_records_with_H_above_22.csv"
df = pd.read_csv(input_path)

print("Records loaded:", len(df))

# --------------------------------------------------
# 2. Function to extract a physical parameter
# --------------------------------------------------

def get_parameter(data, parameter_name):
    phys = data.get("phys_par", [])

    for item in phys:
        if item.get("name") == parameter_name:
            return {
                "value": item.get("value"),
                "sigma": item.get("sigma"),
                "ref": item.get("ref"),
                "notes": item.get("notes"),
                "title": item.get("title"),
                "desc": item.get("desc")
            }

    return {
        "value": None,
        "sigma": None,
        "ref": None,
        "notes": None,
        "title": None,
        "desc": None
    }

# --------------------------------------------------
# 3. Query JPL SBDB for each asteroid
# --------------------------------------------------

results = []

for index, row in df.iterrows():

    spkid = str(row["spkid"])

    api_url = "https://ssd-api.jpl.nasa.gov/sbdb.api"

    params = {
        "spk": spkid,
        "full-prec": "true",
        "phys-par": "1"
    }

    try:
        response = requests.get(api_url, params=params, timeout=30)
        response.raise_for_status()

        data = response.json()

        obj = data.get("object", {})
        orbit = data.get("orbit", {})

        H_info = get_parameter(data, "H")

        results.append({
            "spkid": spkid,
            "original_name": row.get("full_name"),
            "JPL_name": obj.get("fullname"),
            "JPL_PHA": obj.get("pha"),
            "JPL_NEO": obj.get("neo"),
            "JPL_kind": obj.get("kind"),

            "Original_H": row.get("H"),
            "JPL_H": H_info["value"],
            "H_sigma": H_info["sigma"],
            "H_reference": H_info["ref"],
            "H_notes": H_info["notes"],
            "H_title": H_info["title"],
            "H_description": H_info["desc"],

            "Original_MOID": row.get("moid"),
            "JPL_MOID": orbit.get("moid"),

            "Orbit_ID": orbit.get("orbit_id"),
            "Orbit_solution_date": orbit.get("soln_date"),
            "Orbit_source": orbit.get("source"),
            "Orbit_producer": orbit.get("producer"),
            "Condition_code": orbit.get("condition_code"),
            "Covariance_epoch": orbit.get("cov_epoch"),

            "API_status": "Success"

        })

        print(f"Checked {index + 1}/{len(df)}: {spkid}")

    except Exception as e:

        results.append({
            "spkid": spkid,
            "original_name": row.get("full_name"),
            "API_status": f"Failed: {str(e)}"
        })

        print(f"Failed: {spkid}")

    time.sleep(0.15)

# --------------------------------------------------
# 4. Save detailed results
# --------------------------------------------------

result_df = pd.DataFrame(results)

output_dir = "/content/neo_evaluation_results"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "pha_33_records_jpl_detailed_check.csv"
)

result_df.to_csv(output_path, index=False)

# --------------------------------------------------
# 5. Display important findings
# --------------------------------------------------

print("\n========== VERIFICATION SUMMARY ==========")

print("Total records:", len(result_df))
print("Successful API checks:",
      (result_df["API_status"] == "Success").sum())

print("\nJPL PHA flags:")
print(result_df["JPL_PHA"].value_counts(dropna=False))

print("\nH values and uncertainties:")
display(
    result_df[
        [
            "spkid",
            "JPL_name",
            "JPL_PHA",
            "Original_H",
            "JPL_H",
            "H_sigma",
            "H_reference",
            "H_notes"
        ]
    ]
)

print("\nOrbit solution information:")
display(
    result_df[
        [
            "spkid",
            "Orbit_ID",
            "Orbit_solution_date",
            "Orbit_source",
            "Orbit_producer",
            "Condition_code"
        ]
    ]
)

print("\nSaved file:", output_path)



Records loaded: 33
Checked 1/33: 20154019
Checked 2/33: 20468468
Checked 3/33: 20513312
Checked 4/33: 50092240
Checked 5/33: 50020999
Checked 6/33: 50025789
Checked 7/33: 50066102
Checked 8/33: 50063081
Checked 9/33: 50092299
Checked 10/33: 50102706
Checked 11/33: 50102717
Checked 12/33: 50102765
Checked 13/33: 50144173
Checked 14/33: 50180176
Checked 15/33: 50251735
Checked 16/33: 50373705
Checked 17/33: 50399604
Checked 18/33: 50402154
Checked 19/33: 50407631
Checked 20/33: 50423047
Checked 21/33: 50557508
Checked 22/33: 50672584
Checked 23/33: 50699926
Checked 24/33: 50740517
Checked 25/33: 50752803
Checked 26/33: 50774637
Checked 27/33: 50789632
Checked 28/33: 50835532
Checked 29/33: 50838286
Checked 30/33: 54235530
Checked 31/33: 54278314
Checked 32/33: 54336235
Checked 33/33: 54415300

========== VERIFICATION SUMMARY ==========
Total records: 33
Successful API checks: 33

JPL PHA flags:
JPL_PHA
True    33
Name: count, dtype: int64

H values and uncertainties:


,spkid,JPL_name,JPL_PHA,Original_H,JPL_H,H_sigma,H_reference,H_notes
0,20154019,154019 (2002 CZ9),True,22.07,22.07,None,MPO694396,None
1,20468468,468468 (2004 KH17),True,22.01,22.01,None,E2025L08,None
2,20513312,513312 (2007 DM41),True,22.06,22.06,None,MPO905192,None
3,50092240,(1999 FR19),True,22.49,22.49,None,MPO972135,None
4,50020999,(1999 JZ10),True,22.22,22.22,None,MPO739983,None
5,50025789,(2000 AA6),True,22.44,22.44,None,E2025R78,None
6,50066102,(2000 SL10),True,22.16,22.16,None,E2023-XR9,None
7,50063081,(2000 WC1),True,22.57,22.57,None,E2025-CG0,None
8,50092299,(2000 WH10),True,22.42,22.42,.47426,MPO733997,autocmod 2.9c
9,50102706,(2001 XX4),True,22.10,22.1,None,MPO74111,None



Orbit solution information:


,spkid,Orbit_ID,Orbit_solution_date,Orbit_source,Orbit_producer,Condition_code
0,20154019,15,2026-06-06 22:49:52,JPL,Otto Matic,1
1,20468468,64,2025-06-02 06:19:04,JPL,Otto Matic,0
2,20513312,51,2025-03-11 06:22:28,JPL,Otto Matic,0
3,50092240,18,2025-11-01 06:19:45,JPL,Otto Matic,0
4,50020999,12,2021-04-14 20:50:29,JPL,Otto Matic,7
5,50025789,34,2023-03-01 05:59:11,JPL,Otto Matic,0
6,50066102,13,2021-04-14 20:58:36,JPL,Otto Matic,6
7,50063081,18,2025-02-11 05:49:20,JPL,Otto Matic,0
8,50092299,13,2021-04-14 21:00:49,JPL,Otto Matic,9
9,50102706,13,2021-04-14 21:15:54,JPL,Otto Matic,1



Saved file: /content/neo_evaluation_results/pha_33_records_jpl_detailed_check.csv


In [13]:
import pandas as pd
import numpy as np

# Load the detailed JPL verification CSV
file_path = "/content/neo_evaluation_results/pha_33_records_jpl_detailed_check.csv"

df = pd.read_csv(file_path)

# Convert H and uncertainty columns to numeric
df["JPL_H"] = pd.to_numeric(df["JPL_H"], errors="coerce")
df["H_sigma"] = pd.to_numeric(df["H_sigma"], errors="coerce")

# Calculate lower and upper H limits
df["H_lower"] = df["JPL_H"] - df["H_sigma"]
df["H_upper"] = df["JPL_H"] + df["H_sigma"]

# Check whether the uncertainty range includes H = 22
df["Includes_H_22"] = (
    (df["H_lower"] <= 22) &
    (df["H_upper"] >= 22)
)

# Display the results
print("TOTAL RECORDS:", len(df))
print("Records with H uncertainty:", df["H_sigma"].notna().sum())
print("Records without H uncertainty:", df["H_sigma"].isna().sum())

print("\nUNCERTAINTY ANALYSIS:")
display(
    df[
        [
            "spkid",
            "JPL_name",
            "JPL_PHA",
            "JPL_H",
            "H_sigma",
            "H_lower",
            "H_upper",
            "Includes_H_22"
        ]
    ]
)

print("\nSUMMARY:")
print(
    "Uncertainty ranges including H=22:",
    df["Includes_H_22"].sum()
)

print(
    "Uncertainty ranges not including H=22:",
    (
        df["H_sigma"].notna() &
        ~df["Includes_H_22"]
    ).sum()
)

# Save the analysis
output_path = "/content/neo_evaluation_results/pha_33_H_uncertainty_analysis.csv"
df.to_csv(output_path, index=False)

print("\nSaved:", output_path)

TOTAL RECORDS: 33
Records with H uncertainty: 8
Records without H uncertainty: 25

UNCERTAINTY ANALYSIS:


,spkid,JPL_name,JPL_PHA,JPL_H,H_sigma,H_lower,H_upper,Includes_H_22
0,20154019,154019 (2002 CZ9),True,22.07,NaN,NaN,NaN,False
1,20468468,468468 (2004 KH17),True,22.01,NaN,NaN,NaN,False
2,20513312,513312 (2007 DM41),True,22.06,NaN,NaN,NaN,False
3,50092240,(1999 FR19),True,22.49,NaN,NaN,NaN,False
4,50020999,(1999 JZ10),True,22.22,NaN,NaN,NaN,False
5,50025789,(2000 AA6),True,22.44,NaN,NaN,NaN,False
6,50066102,(2000 SL10),True,22.16,NaN,NaN,NaN,False
7,50063081,(2000 WC1),True,22.57,NaN,NaN,NaN,False
8,50092299,(2000 WH10),True,22.42,0.47426,21.94574,22.89426,True
9,50102706,(2001 XX4),True,22.10,NaN,NaN,NaN,False



SUMMARY:
Uncertainty ranges including H=22: 8
Uncertainty ranges not including H=22: 0

Saved: /content/neo_evaluation_results/pha_33_H_uncertainty_analysis.csv


In [14]:
import pandas as pd
import requests
import os

# --------------------------------------------------
# 1. LOAD THE ORIGINAL 33 RECORDS FROM GITHUB
# --------------------------------------------------

source_path = "/content/neo_evaluation_results/pha_33_records_jpl_detailed_check.csv"

original = pd.read_csv(source_path)

original["spkid"] = original["spkid"].astype(str)

print("Original records loaded:", len(original))


# --------------------------------------------------
# 2. QUERY JPL SBDB QUERY API
# --------------------------------------------------

api_url = "https://ssd-api.jpl.nasa.gov/sbdb_query.api"

params = {
    "sb-group": "pha",
    "sb-kind": "a",
    "fields": "spkid,full_name,kind,neo,pha,moid,H,H_sigma,orbit_id,soln_date,condition_code",
    "full-prec": "true"
}

response = requests.get(
    api_url,
    params=params,
    timeout=60
)

response.raise_for_status()

data = response.json()

if "data" not in data or "fields" not in data:
    raise ValueError(f"Unexpected JPL API response: {data}")

pha_list = pd.DataFrame(
    data["data"],
    columns=data["fields"]
)

pha_list["spkid"] = pha_list["spkid"].astype(str)

print("\nCurrent asteroid PHA records returned:", len(pha_list))


# --------------------------------------------------
# 3. CHECK WHICH OF THE 33 RECORDS APPEAR IN THE API
# --------------------------------------------------

pha_ids = set(pha_list["spkid"])

comparison = original.copy()

comparison["Found_in_Query_API"] = (
    comparison["spkid"].isin(pha_ids)
)

# Add matching Query API details
query_details = pha_list.add_suffix("_query")

comparison = comparison.merge(
    query_details,
    left_on="spkid",
    right_on="spkid_query",
    how="left"
)


# --------------------------------------------------
# 4. DISPLAY SUMMARY
# --------------------------------------------------

found_count = comparison["Found_in_Query_API"].sum()
not_found_count = len(comparison) - found_count

print("\n========== JPL QUERY API CROSS-CHECK ==========")

print("Original records:", len(original))
print("Found in Query API:", found_count)
print("Not found in Query API:", not_found_count)

print("\nQuery API PHA flags:")
print(comparison["pha_query"].value_counts(dropna=False))


# --------------------------------------------------
# 5. DISPLAY COMPARISON TABLE
# --------------------------------------------------

print("\n========== DETAILED COMPARISON ==========")

display(
    comparison[
        [
            "spkid",
            "JPL_name",
            "JPL_PHA",
            "Found_in_Query_API",
            "full_name_query",
            "pha_query",
            "H_query",
            "moid_query",
            "orbit_id_query"
        ]
    ]
)


# --------------------------------------------------
# 6. SAVE RESULTS
# --------------------------------------------------

output_dir = "/content/neo_evaluation_results"

os.makedirs(
    output_dir,
    exist_ok=True
)

output_path = os.path.join(
    output_dir,
    "pha_33_query_api_crosscheck.csv"
)

comparison.to_csv(
    output_path,
    index=False
)

print("\nCSV saved successfully:")
print(output_path)


# --------------------------------------------------

# --------------------------------------------------



Original records loaded: 33

Current asteroid PHA records returned: 2551

========== JPL QUERY API CROSS-CHECK ==========
Original records: 33
Found in Query API: 33
Not found in Query API: 0

Query API PHA flags:
pha_query
Y    33
Name: count, dtype: int64

========== DETAILED COMPARISON ==========


,spkid,JPL_name,JPL_PHA,Found_in_Query_API,full_name_query,pha_query,H_query,moid_query,orbit_id_query
0,20154019,154019 (2002 CZ9),True,True,154019 (2002 CZ9),Y,22.07,.0256281,JPL 15
1,20468468,468468 (2004 KH17),True,True,468468 (2004 KH17),Y,22.01,.00344174,64
2,20513312,513312 (2007 DM41),True,True,513312 (2007 DM41),Y,22.06,.0074645,51
3,50092240,(1999 FR19),True,True,(1999 FR19),Y,22.49,.0348184,18
4,50020999,(1999 JZ10),True,True,(1999 JZ10),Y,22.22,.0378895,12
5,50025789,(2000 AA6),True,True,(2000 AA6),Y,22.44,.00770636,34
6,50066102,(2000 SL10),True,True,(2000 SL10),Y,22.16,.0108875,13
7,50063081,(2000 WC1),True,True,(2000 WC1),Y,22.57,.00550594,18
8,50092299,(2000 WH10),True,True,(2000 WH10),Y,22.42,.0489846,JPL 13
9,50102706,(2001 XX4),True,True,(2001 XX4),Y,22.1,.0113228,13



CSV saved successfully:
/content/neo_evaluation_results/pha_33_query_api_crosscheck.csv


In [15]:
import pandas as pd
import numpy as np
import os

# 1. Load the verified 33-record file from GitHub
crosscheck_path = "/content/neo_evaluation_results/pha_33_query_api_crosscheck.csv"

df = pd.read_csv(crosscheck_path)

# 2. Create consolidated audit columns
audit = pd.DataFrame()

audit["SPKID"] = df["spkid"]
audit["Asteroid_Name"] = df["JPL_name"]
audit["Original_H"] = df["Original_H"]
audit["JPL_H"] = df["JPL_H"]
audit["H_sigma"] = pd.to_numeric(df["H_sigma"], errors="coerce")
audit["Original_MOID"] = df["Original_MOID"]
audit["JPL_MOID"] = df["JPL_MOID"]

audit["Individual_API_PHA"] = df["JPL_PHA"]
audit["Query_API_PHA"] = df["pha_query"]
audit["Query_API_NEO"] = df["neo_query"]
audit["Found_in_Current_PHA_Query"] = df["Found_in_Query_API"]

audit["Orbit_ID"] = df["Orbit_ID"]
audit["Orbit_Solution_Date"] = df["Orbit_solution_date"]

# 3. Calculate H uncertainty intervals where available
audit["H_lower"] = audit["JPL_H"] - audit["H_sigma"]
audit["H_upper"] = audit["JPL_H"] + audit["H_sigma"]

audit["H_uncertainty_available"] = audit["H_sigma"].notna()

audit["H22_within_reported_interval"] = np.where(
    audit["H_sigma"].notna(),
    (audit["H_lower"] <= 22) & (audit["H_upper"] >= 22),
    np.nan
)

# 4. Add a clear interpretation status
audit["Uncertainty_Interpretation"] = np.select(
    [
        audit["H_sigma"].isna(),
        audit["H22_within_reported_interval"] == True,
        audit["H22_within_reported_interval"] == False
    ],
    [
        "Uncertainty not reported",
        "Reported interval includes H=22",
        "Reported interval does not include H=22"
    ],
    default="Not determined"
)

# 5. Print summary
print("========== CONSOLIDATED 33-RECORD AUDIT ==========")

print("Total records:", len(audit))

print(
    "Individual API PHA=True:",
    (audit["Individual_API_PHA"] == True).sum()
)

print(
    "Query API PHA=Y:",
    (audit["Query_API_PHA"] == "Y").sum()
)

print(
    "Found in current PHA Query API:",
    (audit["Found_in_Current_PHA_Query"] == True).sum()
)

print(
    "Records with H uncertainty:",
    audit["H_uncertainty_available"].sum()
)

print(
    "Records without H uncertainty:",
    (~audit["H_uncertainty_available"]).sum()
)

print(
    "Reported uncertainty intervals including H=22:",
    (audit["H22_within_reported_interval"] == True).sum()
)

# 6. Display the audit
display(audit)

# 7. Save consolidated audit
output_dir = "/content/neo_evaluation_results"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "pha_33_consolidated_audit.csv"
)

audit.to_csv(output_path, index=False)

print("\nSaved:", output_path)



========== CONSOLIDATED 33-RECORD AUDIT ==========
Total records: 33
Individual API PHA=True: 33
Query API PHA=Y: 33
Found in current PHA Query API: 33
Records with H uncertainty: 8
Records without H uncertainty: 25
Reported uncertainty intervals including H=22: 8


,SPKID,Asteroid_Name,Original_H,JPL_H,H_sigma,Original_MOID,JPL_MOID,Individual_API_PHA,Query_API_PHA,Query_API_NEO,Found_in_Current_PHA_Query,Orbit_ID,Orbit_Solution_Date,H_lower,H_upper,H_uncertainty_available,H22_within_reported_interval,Uncertainty_Interpretation
0,20154019,154019 (2002 CZ9),22.07,22.07,NaN,0.02560,0.025628,True,Y,Y,True,15,2026-06-06 22:49:52,NaN,NaN,False,NaN,Uncertainty not reported
1,20468468,468468 (2004 KH17),22.01,22.01,NaN,0.00344,0.003442,True,Y,Y,True,64,2025-06-02 06:19:04,NaN,NaN,False,NaN,Uncertainty not reported
2,20513312,513312 (2007 DM41),22.06,22.06,NaN,0.00746,0.007464,True,Y,Y,True,51,2025-03-11 06:22:28,NaN,NaN,False,NaN,Uncertainty not reported
3,50092240,(1999 FR19),22.49,22.49,NaN,0.03480,0.034818,True,Y,Y,True,18,2025-11-01 06:19:45,NaN,NaN,False,NaN,Uncertainty not reported
4,50020999,(1999 JZ10),22.22,22.22,NaN,0.03790,0.037889,True,Y,Y,True,12,2021-04-14 20:50:29,NaN,NaN,False,NaN,Uncertainty not reported
5,50025789,(2000 AA6),22.44,22.44,NaN,0.00771,0.007706,True,Y,Y,True,34,2023-03-01 05:59:11,NaN,NaN,False,NaN,Uncertainty not reported
6,50066102,(2000 SL10),22.16,22.16,NaN,0.01090,0.010887,True,Y,Y,True,13,2021-04-14 20:58:36,NaN,NaN,False,NaN,Uncertainty not reported
7,50063081,(2000 WC1),22.57,22.57,NaN,0.00551,0.005506,True,Y,Y,True,18,2025-02-11 05:49:20,NaN,NaN,False,NaN,Uncertainty not reported
8,50092299,(2000 WH10),22.42,22.42,0.47426,0.04900,0.048985,True,Y,Y,True,13,2021-04-14 21:00:49,21.94574,22.89426,True,1.0,Reported interval includes H=22
9,50102706,(2001 XX4),22.10,22.10,NaN,0.01130,0.011323,True,Y,Y,True,13,2021-04-14 21:15:54,NaN,NaN,False,NaN,Uncertainty not reported



Saved: /content/neo_evaluation_results/pha_33_consolidated_audit.csv


In [16]:
# ============================================================
# FINAL EXPORT: PACKAGE ALL PART 1 + PART 2 CSV OUTPUTS
# ============================================================
import os
import zipfile
import pandas as pd
from google.colab import files

OUTPUT_DIR = "/content/neo_evaluation_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Create a manifest so every generated file is easy to identify.
manifest_rows = []
for filename in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, filename)
    if os.path.isfile(filepath) and filename.lower().endswith(".csv"):
        manifest_rows.append({
            "filename": filename,
            "size_bytes": os.path.getsize(filepath),
            "github_destination": "results/analysis/"
        })

manifest = pd.DataFrame(manifest_rows)
manifest_path = os.path.join(OUTPUT_DIR, "output_manifest.csv")
manifest.to_csv(manifest_path, index=False)

zip_path = "/content/NEO_Part1_Part2_All_Results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in sorted(os.listdir(OUTPUT_DIR)):
        filepath = os.path.join(OUTPUT_DIR, filename)
        if os.path.isfile(filepath) and filename.lower().endswith(".csv"):
            zf.write(filepath, arcname=filename)

print("CSV files included:", len(manifest))
display(manifest)
print("\nAll results are in:", OUTPUT_DIR)
print("Downloading one ZIP containing every generated CSV...")
files.download(zip_path)


CSV files included: 29


,filename,size_bytes,github_destination
0,all_methods_metrics.csv,3034,results/analysis/
1,all_methods_predictions.csv,1776917,results/analysis/
2,discovery_era_diameter_availability.csv,651,results/analysis/
3,discovery_era_pha_rates.csv,1032,results/analysis/
4,full_test_set_metrics.csv,1039,results/analysis/
5,h_category_pha_rates.csv,281,results/analysis/
6,h_group_diameter_availability.csv,356,results/analysis/
7,h_group_pha_rates.csv,505,results/analysis/
8,measured_diameter_metrics.csv,1169,results/analysis/
9,missing_diameter_metrics.csv,1056,results/analysis/



All results are in: /content/neo_evaluation_results


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>